In [1]:
import csv
from pathlib import Path

In [2]:
actual_path = Path("raw-zip-actual")
actual_files = sorted(actual_path.glob("*.zip"), reverse=True)
print(len(actual_files))
actual_files[:5]

230


[PosixPath('raw-zip-actual/20260201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20260101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251001RTLineOutages_csv.zip')]

In [3]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"), reverse=True)
print(len(scheduled_files))
scheduled_files[:5]

218


[PosixPath('raw-zip-scheduled/20260201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20260101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251001SCLineOutages_csv.zip')]

In [4]:
import re
from collections import namedtuple

ActualOutage = namedtuple(
    "ActualOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "outage_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)

ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)


equipment_name_pattern = r"^([A-Za-z0-9._]{8})-([A-Za-z0-9._]{8})_(\d{2,3})_(.+)$"

In [5]:
zip_path = actual_files[0]

In [7]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_actual_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ActualOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        outage_datetime=datetime.strptime(row["Outage Date/Time"], "%m/%d/%Y %H:%M:%S"),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name, parse_row):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        data = [parse_row(row) for row in csv_reader]
        print(len(data))
        data = [row for row in data if row is not None]
        print(len(data))
    return data


# Example using an existing variable in the notebook:
zip_path = actual_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member, parse_actual_outage)

raw-zip-actual/20260201RTLineOutages_csv.zip
members: ['20260201RTLineOutages.csv', '20260202RTLineOutages.csv', '20260203RTLineOutages.csv', '20260204RTLineOutages.csv', '20260205RTLineOutages.csv', '20260206RTLineOutages.csv', '20260207RTLineOutages.csv', '20260208RTLineOutages.csv', '20260209RTLineOutages.csv', '20260210RTLineOutages.csv', '20260211RTLineOutages.csv', '20260212RTLineOutages.csv', '20260213RTLineOutages.csv', '20260214RTLineOutages.csv', '20260215RTLineOutages.csv', '20260216RTLineOutages.csv', '20260217RTLineOutages.csv', '20260218RTLineOutages.csv', '20260219RTLineOutages.csv', '20260220RTLineOutages.csv', '20260221RTLineOutages.csv', '20260222RTLineOutages.csv', '20260223RTLineOutages.csv', '20260224RTLineOutages.csv']
45505
19148


# Compress a csv

In [20]:
data[:10]

[ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25013, equipment_name='E.SAYRE_-NWAVERLY_115_956', outage_datetime=datetime.datetime(2025, 6, 2, 8, 0)),
 ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25015, equipment_name='WARREN__-FALCONER_115_171', outage_datetime=datetime.datetime(2025, 9, 4, 15, 16)),
 ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25020, equipment_name='HUDSONP_-FARRAGUT_345_B3402', outage_datetime=datetime.datetime(2018, 1, 15, 10, 15)),
 ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25033, equipment_name='LONG_MTN-CRICKVLY_345_398', outage_datetime=datetime.datetime(2025, 10, 22, 1, 17)),
 ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25038, equipment_name='MARION__-FARRAGUT_345_C3403', outage_datetime=datetime.datetime(2018, 1, 15, 12, 5)),
 ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 0, 2), ptid=25043, equipment_name='FARRAGUT_345C_345A_TR12', outage_datetime=

In [7]:
from tqdm import tqdm

actual_outages = []
for zip_path in tqdm(actual_files):
    for member in list_csvs(zip_path):
        rows = read_csv_from_zip(zip_path, member, parse_actual_outage)
        actual_outages.extend(rows)

 20%|██        | 47/230 [08:55<34:45, 11.40s/it]


KeyboardInterrupt: 